# Structured Outputs

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `output_type` to get typed, structured responses from agents using Pydantic models, dataclasses, and TypedDict.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Structured Output with Pydantic

Set `output_type` to a Pydantic `BaseModel`. The agent returns a typed object instead of free-form text.

In [ ]:
from pydantic import BaseModel
from agents import Agent, Runner


class CalendarEvent(BaseModel):
    title: str
    date: str
    time: str
    duration_minutes: int
    participants: list[str]


agent = Agent(
    name="Event Extractor",
    instructions="Extract calendar event details from the user's message. Always respond with structured data.",
    output_type=CalendarEvent,
)

result = Runner.run_sync(
    agent,
    "Schedule a team standup tomorrow at 9am for 30 minutes with Alice and Bob.",
)

print(f"Title: {result.final_output.title}")
print(f"Date: {result.final_output.date}")
print(f"Time: {result.final_output.time}")
print(f"Duration: {result.final_output.duration_minutes} minutes")
print(f"Participants: {result.final_output.participants}")
print(f"\nType: {type(result.final_output)}")

## 4. Using Field Descriptions

Pydantic `Field` descriptions guide the model on what each field should contain.

In [ ]:
from pydantic import Field


class SentimentResult(BaseModel):
    text: str = Field(description="The analyzed text")
    sentiment: str = Field(description="positive, negative, or neutral")
    confidence: float = Field(description="Confidence score between 0 and 1")
    key_phrases: list[str] = Field(
        description="Important phrases that influenced the sentiment"
    )


analyzer = Agent(
    name="Sentiment Analyzer",
    instructions="Analyze the sentiment of the provided text.",
    output_type=SentimentResult,
)

result = Runner.run_sync(
    analyzer, "I absolutely loved the new restaurant! The food was incredible."
)

print(f"Sentiment: {result.final_output.sentiment}")
print(f"Confidence: {result.final_output.confidence}")
print(f"Key phrases: {result.final_output.key_phrases}")

## 5. Nested Output Models

Output types can include nested Pydantic models for complex structures.

In [ ]:
class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str
    address: Address


extractor = Agent(
    name="Contact Extractor",
    instructions="Extract contact information from the provided text.",
    output_type=ContactInfo,
)

result = Runner.run_sync(
    extractor,
    "John Smith, john@example.com, 555-1234, lives at 123 Main St, Springfield, IL 62701",
)

print(f"Name: {result.final_output.name}")
print(f"Email: {result.final_output.email}")
print(f"Phone: {result.final_output.phone}")
print(f"City: {result.final_output.address.city}")
print(f"State: {result.final_output.address.state}")

## 6. Dataclass as Output Type

The SDK also supports `dataclass` for simpler cases where full Pydantic validation isn't needed.

In [ ]:
from dataclasses import dataclass


@dataclass
class MovieReview:
    title: str
    rating: int
    summary: str


review_agent = Agent(
    name="Review Parser",
    instructions="Parse movie reviews into structured data.",
    output_type=MovieReview,
)

result = Runner.run_sync(
    review_agent,
    "Inception is a masterpiece. 9/10. A mind-bending thriller about dreams within dreams.",
)

print(f"Title: {result.final_output.title}")
print(f"Rating: {result.final_output.rating}/10")
print(f"Summary: {result.final_output.summary}")

## Key Takeaways

- Set `output_type` on an Agent to get structured, typed responses instead of free-form text
- `result.final_output` is a Python object you can access with dot notation
- Pydantic `BaseModel` is the recommended approach — it supports validation, defaults, and field descriptions
- Nested models work for complex output structures
- `dataclass` and `TypedDict` are also supported as alternatives